In [ ]:
import torch
import numpy as np
import torchvision
import torch.nn as nn
from torchvision.datasets import mnist

In [ ]:
transform = torchvision.transforms.Compose([
    # CIFAR-10 images are 3 channels, we might need to adjust the VAE input dimension later
    torchvision.transforms.Grayscale(num_output_channels=1), # Keep as 3 channels for now
    torchvision.transforms.ToTensor(),
    # Removed normalization to keep data in [0, 1] range for consistency with Noising function and loss
    # torchvision.transforms.Normalize((0.5,), (0.5,))
])

# Load the full CIFAR-10 training data
full_train_data = torchvision.datasets.CIFAR10(root="./data", train=True, transform=transform, download=True)

# Filter the dataset to include only images with a specific label (e.g., label 3 for cat)
# You can change the label number to train on a different class
target_label = 3
seven_indices = [i for i, (image, label) in enumerate(full_train_data) if label == target_label]
train_data_seven = torch.utils.data.Subset(full_train_data, seven_indices)

# Use the filtered dataset for training
train_data = train_data_seven

100%|██████████| 170M/170M [00:04<00:00, 41.9MB/s]


In [ ]:
transform = torchvision.transforms.Compose([
    torchvision.transforms.Grayscale(num_output_channels=1),
    torchvision.transforms.ToTensor(),
    # Removed normalization to keep data in [0, 1] range for consistency with Noising function and loss
    transforms.Normalize(mean=[0.0, 0.0, 0.0], std=[1.0, 1.0, 1.0])
])

# Load the full MNIST training data
full_train_data = mnist.MNIST(root="./data", train=True, transform=transform, download=True)

# Filter the dataset to include only images with label 7
seven_indices = [i for i, (image, label) in enumerate(full_train_data) if label == 7]
train_data_seven = torch.utils.data.Subset(full_train_data, seven_indices)

# Use the filtered dataset for training
train_data = train_data_seven

100%|██████████| 9.91M/9.91M [00:00<00:00, 18.3MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 497kB/s]
100%|██████████| 1.65M/1.65M [00:00<00:00, 4.56MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 8.41MB/s]


In [ ]:
def Noising(or_images, num_steps=1000, theta = 4, bias= 10):
    noisy_image = or_images
    for i in range(num_steps):
        noisy_image = theta * noisy_image * (1 - noisy_image)
        noisy_image = torch.clamp(noisy_image, 0, 1)
    noise_added = noisy_image - or_images
    return noisy_image + bias, noise_added

In [ ]:
import torch.nn as nn
import torch.nn.functional as F
import torch

class VAE(nn.Module):
    def __init__(self, input_dim, hidden_dim, latent_dim, img_size=32):
        super(VAE, self).__init__()

        self.input_dim = input_dim
        self.img_size = img_size   # for reshaping back into images

        # ---------------------- Encoder ----------------------
        self.fc1 = nn.Linear(input_dim, hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, hidden_dim)
        self.fc3 = nn.Linear(hidden_dim, hidden_dim)

        self.fc_mu = nn.Linear(hidden_dim, latent_dim)
        self.fc_logvar = nn.Linear(hidden_dim, latent_dim)

        # ---------------------- Decoder ----------------------
        self.fc4 = nn.Linear(latent_dim, hidden_dim)
        self.fc5 = nn.Linear(hidden_dim, hidden_dim)
        self.fc6 = nn.Linear(hidden_dim, hidden_dim)
        self.fc7 = nn.Linear(hidden_dim, input_dim)   # returns flat vector

        # ---------------------- CNN Decoder Refinement ----------------------
        self.cnn1 = nn.Conv2d(1, 8, kernel_size=3, stride=1, padding=1)
        self.cnn2 = nn.Conv2d(8,16, kernel_size=3, stride=1, padding=1)
        self.cnn3 = nn.Conv2d(16,8, kernel_size=3, stride=1, padding=1)
        self.cnn4 = nn.Conv2d(8,1, kernel_size=3, stride=1, padding=1)


    # ========================================================
    # ---------------------- Encoder -------------------------
    def encode(self, x):
        h = F.relu(self.fc1(x))
        h = F.relu(self.fc2(h))
        h = F.relu(self.fc3(h))
        return h, self.fc_mu(h), self.fc_logvar(h)

    # ------------------- Reparameterization -----------------
    def reparameterize(self, mu, logvar):
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + eps * std

    # ---------------------- Decoder -------------------------
    def decode(self, z):
        h = F.relu(self.fc4(z))
        h = F.relu(self.fc5(h))
        h = F.relu(self.fc6(h))

        out = torch.sigmoid(self.fc7(h)) # flat vector = B × input_dim

        # reshape into (B,1,H,W) -> then refine using CNN
        img = out.view(-1, 1, self.img_size, self.img_size)

        img = F.relu(self.cnn1(img))
        img = F.relu(self.cnn2(img))
        img = F.relu(self.cnn3(img))
        img = torch.sigmoid(self.cnn4(img))  # final reconstructed image

        return out

    # ---------------------- Forward -------------------------
    def forward(self, x):
        mu, logvar = self.encode(x.view(-1, self.input_dim))
        z = self.reparameterize(mu, logvar)
        recon = self.decode(z)
        return recon, mu, logvar

In [ ]:
# train_ddpm_liquid.py
# Full DDPM training + sampling using LiquidDenoiserUNet
# Requires: torch, torchvision

import os
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import transforms, datasets, utils
from torch.utils.data import DataLoader
from tqdm import tqdm

# -------------------------
# Hyperparameters / Config
# -------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
seed = 42
torch.manual_seed(seed)

image_size = 32               # CIFAR-10: 32x32
in_channels = 3
batch_size = 128
epochs = 15                   # reduce for quick tests
lr = 0.01
weight_decay = 0.0
ema_decay = 0.995
save_every = 5
out_dir = "./ddpm_liquid_out"
os.makedirs(out_dir, exist_ok=True)

timesteps = 1000              # diffusion timesteps (T)
device = torch.device(device)

# # -------------------------
# # Diffusion schedule utils
# # -------------------------
# def linear_beta_schedule(timesteps, beta_start=1e-4, beta_end=2e-2):
#     return torch.linspace(beta_start, beta_end, timesteps)

# betas = linear_beta_schedule(timesteps=timesteps).to(device)          # (T,)
# alphas = 1.0 - betas
# alphas_cumprod = torch.cumprod(alphas, dim=0)
# alphas_cumprod_prev = torch.cat([torch.tensor([1.0], device=device), alphas_cumprod[:-1]], dim=0)

# sqrt_alphas_cumprod = torch.sqrt(alphas_cumprod)
# sqrt_one_minus_alphas_cumprod = torch.sqrt(1.0 - alphas_cumprod)
# posterior_variance = betas * (1.0 - alphas_cumprod_prev) / (1.0 - alphas_cumprod)

# # helper to fetch tensors for batch of t
# def extract(a, t, x_shape):
#     # a: (T,)   t: (B,)   -> returns (B,1,1,1)
#     out = a.gather(-1, t).float().reshape(t.shape[0], *((1,) * (len(x_shape) - 1)))
#     return out

# -------------------------
# Model (Liquid UNet adapted for timestep-conditioning and predicting noise)
# -------------------------
class LiquidBlock(nn.Module):
    def __init__(self, channels, inject_channels=None, hidden_channels=None, T_steps=6, dt=1.0):
        super().__init__()
        hidden_channels = hidden_channels or channels
        inject_channels = inject_channels or channels

        self.phi = nn.Sequential(
            nn.GroupNorm(1, channels),
            nn.SiLU(),
            nn.Conv2d(channels, hidden_channels, 3, padding=1),
            nn.SiLU(),
            nn.Conv2d(hidden_channels, channels, 3, padding=1)
        )

        # gate net expects channels + inject_channels input
        self.gate_net = nn.Sequential(
            nn.Conv2d(channels + inject_channels, channels, kernel_size=1),
            nn.SiLU(),
            nn.Conv2d(channels, channels * 2, kernel_size=1),  # outputs a and b
        )

        self.input_proj = nn.Conv2d(inject_channels, channels, kernel_size=1)
        self.T = T_steps
        self.dt = float(dt)

    def forward(self, h, inject):
        # expect inject to be same spatial size as h, and inject_channels matches provided
        for _ in range(self.T):
            g_in = torch.cat([h, inject], dim=1)
            gates = self.gate_net(g_in)
            a, b = gates.chunk(2, dim=1)
            a = torch.sigmoid(a)
            b = torch.sigmoid(b)
            phi_h = self.phi(h)
            inj = self.input_proj(inject)
            dh = - a * h + b * phi_h + inj
            h = h + self.dt * dh
        return h

class LiquidDenoiserUNet(nn.Module):
    def __init__(self, in_ch=3, base_ch=64, ch_mult=(1,2,4), liquid_steps=6, t_emb_dim=256):
        super().__init__()
        self.t_emb_dim = t_emb_dim
        # time embedding
        self.time_mlp = nn.Sequential(
            nn.Linear(t_emb_dim, t_emb_dim*4),
            nn.SiLU(),
            nn.Linear(t_emb_dim*4, t_emb_dim)
        )
        # small sinusoidal embed
        self.t_emb_proj = nn.Linear(t_emb_dim, t_emb_dim)

        # encoder
        enc_chs = []
        prev = in_ch
        self.enc_blocks = nn.ModuleList()
        for m in ch_mult:
            out = base_ch * m
            block = nn.Sequential(
                nn.Conv2d(prev, out, 3, padding=1),
                nn.GroupNorm(1, out),
                nn.SiLU(),
                nn.Conv2d(out, out, 3, padding=1),
                nn.GroupNorm(1, out),
                nn.SiLU()
            )
            self.enc_blocks.append(block)
            enc_chs.append(out)
            prev = out
        self.downs = nn.ModuleList([nn.Conv2d(enc_chs[i], enc_chs[i], 4, 2, 1) for i in range(len(enc_chs))])

        # bottleneck
        bottleneck_ch = enc_chs[-1]
        self.bottleneck_proj = nn.Conv2d(bottleneck_ch, bottleneck_ch, 1)
        # inject_proj will convert (in_ch) image -> bottleneck channels at correct spatial resolution
        self.inject_proj = nn.Sequential(
            nn.Conv2d(in_ch, bottleneck_ch, kernel_size=3, stride=1, padding=1),
            nn.SiLU()
        )
        self.liquid = LiquidBlock(channels=bottleneck_ch, inject_channels=bottleneck_ch, hidden_channels=bottleneck_ch, T_steps=liquid_steps)

        # decoder
        self.up_convs = nn.ModuleList()
        self.dec_blocks = nn.ModuleList()
        rev_chs = list(reversed(enc_chs))
        prev = bottleneck_ch
        for out in rev_chs:
            up = nn.ConvTranspose2d(prev, out, kernel_size=4, stride=2, padding=1)
            self.up_convs.append(up)
            dec_block = nn.Sequential(
                nn.Conv2d(out + out, out, 3, padding=1),
                nn.GroupNorm(1, out),
                nn.SiLU(),
                nn.Conv2d(out, out, 3, padding=1),
                nn.GroupNorm(1, out),
                nn.SiLU()
            )
            self.dec_blocks.append(dec_block)
            prev = out

        # final: predict noise (epsilon) of same shape as input
        self.final = nn.Conv2d(base_ch, in_ch, kernel_size=1)

        # small linear to project sinusoidal into t_emb_dim
        self._sinusoidal_dim = t_emb_dim

    def sinusoidal_embedding(self, t):
        # t: (B,) long
        half = self._sinusoidal_dim // 2
        freq = torch.exp(-math.log(10000) * torch.arange(0, half, dtype=torch.float32, device=t.device) / (half - 1))
        emb = t[:, None].float() * freq[None, :]
        emb = torch.cat([torch.sin(emb), torch.cos(emb)], dim=-1)
        if self._sinusoidal_dim % 2 == 1:
            emb = F.pad(emb, (0,1))
        return emb

    def forward(self, x_t, t):
        """
        x_t: noisy image at timestep t    shape (B, C, H, W)
        t: timesteps tensor (B,)
        returns: predicted noise eps_hat of shape (B,C,H,W)
        """
        # time embedding
        temb = self.sinusoidal_embedding(t)           # (B, t_emb)
        temb = self.t_emb_proj(temb)
        temb = self.time_mlp(temb)                    # (B, t_emb)

        # encoder
        skips = []
        h = x_t
        for block, down in zip(self.enc_blocks, self.downs):
            h = block(h)
            skips.append(h)
            h = down(h)

        # bottleneck: project, prepare inject and run liquid
        h = self.bottleneck_proj(h)   # (B, Cb, Hb, Wb)
        # inject: downsample x_t to match h spatial and project channels
        inject = F.interpolate(x_t, size=h.shape[-2:], mode='bilinear', align_corners=False)
        inject = self.inject_proj(inject)
        # modulate inject with time embedding spatially
        # broadcast temb -> (B, Cb, 1,1)
        temb_proj = temb[:, :, None, None]
        inject = inject + temb_proj

        h = self.liquid(h, inject)

        # decoder
        for up, dec, skip in zip(self.up_convs, self.dec_blocks, reversed(skips)):
            h = up(h)
            if h.shape[-2:] != skip.shape[-2:]:
                h = F.interpolate(h, size=skip.shape[-2:], mode='nearest')
            h = torch.cat([h, skip], dim=1)
            h = dec(h)

        eps_pred = self.final(h)
        return eps_pred

# -------------------------
# EMA helper
# -------------------------
class EMA:
    def __init__(self, model, decay=0.9999, warmup=100):
        self.model = model
        self.decay = decay
        self.shadow = {}
        self.warmup = warmup
        self.num_updates = 0
        # initialize
        for name, p in model.named_parameters():
            if p.requires_grad:
                self.shadow[name] = p.detach().clone()

    @torch.no_grad()
    def update(self, model):
        self.num_updates += 1
        decay = min(self.decay, (1 + self.num_updates) / (10 + self.num_updates))
        for name, p in model.named_parameters():
            if p.requires_grad:
                assert name in self.shadow
                new_val = (1.0 - decay) * p.detach() + decay * self.shadow[name]
                self.shadow[name].copy_(new_val)

    @torch.no_grad()
    def apply_shadow(self, model):
        for name, p in model.named_parameters():
            if p.requires_grad:
                assert name in self.shadow
                p.data.copy_(self.shadow[name])

# -------------------------
# Dataset and loader (CIFAR-10)
# -------------------------
transform = transforms.Compose([
    transforms.Resize(image_size),
    transforms.ToTensor(),                 # [0,1]
    # transforms.Normalize(mean=[0.0, 0.0, 0.0], std=[1.0, 1.0, 1.0])
])
train_ds = datasets.CIFAR10(root="./data", train=True, download=True, transform=transform)
train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, num_workers=4, pin_memory=True)

# -------------------------
# Noise utilities q_sample
# -------------------------
# def q_sample(x0, t, noise=None):
#     """
#     Sample x_t given x0 and timestep t:
#     x_t = sqrt_alpha_cumprod[t]*x0 + sqrt(1 - alpha_cumprod[t]) * noise
#     """
#     if noise is None:
#         noise = torch.randn_like(x0)
#     sqrt_ac = extract(sqrt_alphas_cumprod, t, x0.shape)
#     sqrt_om = extract(sqrt_one_minus_alphas_cumprod, t, x0.shape)
#     return sqrt_ac * x0 + sqrt_om * noise

# # -------------------------
# # p_sample (single step of reverse) and p_sample_loop (full sampling)
# # -------------------------
# @torch.no_grad()
# def p_sample(model, x_t, t):
#     """
#     Compute one reverse step p(x_{t-1} | x_t)
#     using model to predict eps.
#     """
#     bet = extract(betas, t, x_t.shape)
#     sqrt_one_minus_ac = extract(sqrt_one_minus_alphas_cumprod, t, x_t.shape)
#     ac = extract(alphas, t, x_t.shape)
#     ac_prod = extract(alphas_cumprod, t, x_t.shape)
#     model_mean_coef1 = 1.0 / torch.sqrt(ac)
#     model_mean_coef2 = (bet / sqrt_one_minus_ac)
#     eps_pred = model(x_t, t)
#     # predicted x0
#     x0_pred = (x_t - sqrt_one_minus_ac * eps_pred) / torch.sqrt(ac_prod)
#     # clip x0
#     x0_pred = torch.clamp(x0_pred, -1.0, 1.0)

#     # posterior mean
#     mean = model_mean_coef1 * (x_t - model_mean_coef2 * eps_pred)
#     if (t == 0).all():
#         return mean
#     var = extract(posterior_variance, t, x_t.shape)
#     noise = torch.randn_like(x_t)
#     return mean + torch.sqrt(var) * noise

# @torch.no_grad()
# def p_sample_loop(model, shape, device):
#     b = shape[0]
#     img = torch.randn(shape, device=device)
#     for i in reversed(range(timesteps)):
#         t = torch.full((b,), i, dtype=torch.long, device=device)
#         img = p_sample(model, img, t)
    # return img

# -------------------------
# Training loop
# -------------------------
def train():
    model = LiquidDenoiserUNet(in_ch=in_channels, base_ch=64, ch_mult=(1,2,4), liquid_steps=8, t_emb_dim=256).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    ema = EMA(model, decay=ema_decay)

    global_step = 0
    for epoch in range(epochs):
        pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs}")
        for xb, _ in pbar:
            xb = xb.to(device)  # normalized to [-1,1]
            b = xb.shape[0]
            t = torch.randint(0, timesteps, (b,), device=device).long()
            noise = torch.randn_like(xb)
            # print(xb)
            x_t, _ = Noising(xb, num_steps=50, theta=4.0)
            # x_t = q_sample(xb, t, noise=noise)

            eps_pred = model(x_t, t)
            loss = F.mse_loss(eps_pred, noise)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            ema.update(model)

            global_step += 1
            if global_step % 50 == 0:
                pbar.set_postfix(loss=loss.item())

        # Save checkpoint + sample every few epochs
        if (epoch + 1) % save_every == 0 or (epoch == epochs-1):
            ckpt = {
                "model_state": model.state_dict(),
                "optimizer_state": optimizer.state_dict(),
                "epoch": epoch,
                "global_step": global_step,
            }
            torch.save(ckpt, os.path.join(out_dir, f"ckpt_epoch_{epoch+1}.pt"))

            # apply EMA weights, sample, then restore model weights
            orig_state = {k: v.clone() for k, v in model.state_dict().items()}
            ema.apply_shadow(model)
            # samples = p_sample_loop(model, (64, in_channels, image_size, image_size), device)
            # # unnormalize from [-1,1] to [0,1] for grid saving
            # samples = (samples.clamp(-1,1) + 1) / 2.0
            # utils.save_image(samples, os.path.join(out_dir, f"samples_epoch_{epoch+1}.png"), nrow=8)
            # restore
            model.load_state_dict(orig_state)

    # final save
    torch.save(model.state_dict(), os.path.join(out_dir, "model_final.pt"))

if __name__ == "__main__":
    train()


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:627: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(
Epoch 15/15: 100%|██████████| 391/391 [01:04<00:00,  6.06it/s, loss=1]


In [ ]:
from torch.utils.data import DataLoader
import torch.optim as optim
import torch
import random
import torch.nn.functional as F

def train(model, train_data, num_epochs, max_noising_steps=1000, batch_size=64, device="cpu"): # Renamed noising_steps to max_noising_steps
    model.train()
    optimizer = optim.Adam(model.parameters(), lr=0.01)
    scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=4, gamma=0.1)
    train_loader = DataLoader(train_data, batch_size=batch_size, shuffle=True)

    lambda_l1 = 0.0001
    critirion = nn.MSELoss()

    # Ensure model is on the correct device before starting training
    model.to(device)

    # --- Explicitly move linear layer parameters to device ---
    if hasattr(model, 'fc') and model.fc is not None:
        model.fc.to(device)
    # --- End Explicit Move ---


    for epoch in range(num_epochs):
        total_loss = 0.0

        for batch_idx, (images, labels) in enumerate(train_loader):
            images = images.to(device)

            # Generate a sequence of noisy images with decreasing noise levels
            noisy_sequence = []
            original_images_flat = images.view(images.size(0), -1) # Flatten original images once


            noisy_image, _ = Noising(images.clone().to(device), num_steps=max_noising_steps, theta=4.0) # Apply noising
            # print("original:", images.shape, " noisy:", noisy_image.shape)
            # exit()
            #     # Ensure noisy_image is on the correct device before flattening
            #     noisy_sequence.append(noisy_image.to(device).view(images.size(0), -1)) # Flatten and add to sequence


            # # Stack the noisy images along the sequence dimension (batch_size, sequence_length, input_dim)
            # noisy_sequence_tensor = torch.stack(noisy_sequence, dim=1).to(device)


            # Flatten input before feeding to VAE
            noisy_flat = noisy_image.view(noisy_image.size(0), -1)

            # Feed the noisy sequence to the RNN model
            denoised_images_flat, _, _ = model(noisy_image)


            # Calculate the loss (e.g., MSE between denoised image and original image)
            # # Compare the denoised output to the original, clean image
            denoised_images_flat = denoised_images_flat.view(denoised_images_flat.shape[0], -1)
            # original_images_flat = original_images_flat.view(original_images_flat.shape[0], -1)

            loss = critirion(denoised_images_flat, original_images_flat)



            l1_reg = torch.tensor(0.).to(device)
            for param in model.parameters():
                l1_reg += torch.norm(param, 1)

            loss = loss + lambda_l1 * l1_reg


            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            total_loss += loss.item() # Use item() to get the scalar loss value

        scheduler.step()

        avg_loss = total_loss / len(train_loader.dataset) # Calculate average loss per sample
        print(f"Epoch [{epoch+1}/{num_epochs}], Loss: {avg_loss:.8f}")


# Instantiate the RNN model
input_dim = 32 * 32  # CIFAR-10 image dimensions
hidden_dim = 400 # Example hidden dimension
latent_dim = 4 # Example latent dimension
model = VAE(input_dim, hidden_dim, latent_dim)

# Set device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# model.to(device) # Moved inside the train function

# Use train_data which is the CIFAR-10 data filtered for label 3
train(model, train_data=train_data, num_epochs=20, max_noising_steps=10, device=device) # Instantiate and pass model directly with 2 arguments

/usr/local/lib/python3.12/dist-packages/torch/nn/modules/loss.py:634: UserWarning: Using a target size (torch.Size([64, 784])) that is different to the input size (torch.Size([49, 1024])). This will likely lead to incorrect results due to broadcasting. Please ensure they have the same size.
  return F.mse_loss(input, target, reduction=self.reduction)


RuntimeError: The size of tensor a (1024) must match the size of tensor b (784) at non-singleton dimension 1

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

def generate_images_vae(model, num_images_to_generate, latent_dim, device):
    model.eval() # Set the model to evaluation mode
    plt.figure(figsize=(num_images_to_generate * 2, 2))

    with torch.no_grad(): # Disable gradient calculation
        for i in range(num_images_to_generate):
            # Generate a random image from Beta(0.5, 0.5) distribution
            random_image_np = np.random.beta(0.5, 0.5, size=(32, 32))
            random_image_np,_ = Noising(random_image_np, num_steps=100)
            random_image_tensor = torch.tensor(random_image_np, dtype=torch.float32).unsqueeze(0).unsqueeze(0).to(device)

            # Flatten the random image
            random_image_flat = random_image_tensor.view(random_image_tensor.size(0), -1)

            # Pass the flattened random image through the VAE encoder to get mu and logvar
            # mu, logvar = model.encode(random_image_flat)

            # # Reparameterize to get a latent vector
            # latent_vector = model.reparameterize(mu, logvar)

            # # Pass the latent vector through the VAE decoder
            # generated_image_flat = model.decode(latent_vector)

            # Reshape the output from a flattened vector back to an image tensor
            generated_image_flat, _, _ = model(random_image_flat)
            # generated_image_flat = model.cnn_forward(reconstructed_images_linear)

            generated_image_tensor = generated_image_flat.view(1, 1, 32, 32)

            # Detach and move to CPU
            generated_image_np = generated_image_tensor.squeeze().cpu().numpy()
            generated_image_np = generated_image_np

            # Display the generated image
            plt.subplot(1, num_images_to_generate, i + 1)
            plt.imshow(generated_image_np, cmap='gray', vmin=0, vmax=1)
            plt.title(f"Gen {i+1}")
            plt.axis('off')

    plt.tight_layout()
    plt.show()

# Assume vae_model is already trained and available
# Assume latent_dim is defined
# Assume device is defined

# Example usage (assuming vae_model, latent_dim, and device are defined from previous steps):
# generate_images_vae(model=vae_model, num_images_to_generate=10, latent_dim=latent_dim, device=device)

In [ ]:
# Example usage (assuming vae_model, latent_dim, and device are defined from previous steps):

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
generate_images_vae(model=model, num_images_to_generate=20, latent_dim=latent_dim, device=device)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Take a few images from the train_data_seven dataset
num_images_to_show = 20
plt.figure(figsize=(num_images_to_show * 2, 2))

for i in range(num_images_to_show):
    # Access the image and label from the dataset
    image, label = train_data_seven[i+30]

    plt.subplot(1, num_images_to_show, i + 1)
    # Remove the channel dimension for displaying with matplotlib
    plt.imshow(image.squeeze().numpy(), cmap='gray')
    plt.title(f"Label: {label}")
    plt.axis('off')

plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import torch

def denoise_random_image_rnn(model, noising_steps, device): # noising_steps is no longer used for generating sequence
    model.eval() # Set the model to evaluation mode

    plt.figure(figsize=(2, 2)) # Only show original and denoised images

    with torch.no_grad(): # Disable gradient calculation
        # Generate a random image from Beta(0.5, 0.5) distribution (for CIFAR-10 size)
        random_image_np = np.random.beta(0.5, 0.5, size=(32, 32))
        random_image_tensor = torch.tensor(random_image_np, dtype=torch.float32).unsqueeze(0).unsqueeze(0).to(device)

        # Reshape the random image for RNN input (batch_size, sequence_length=1, input_dim)
        rnn_input = random_image_tensor.view(random_image_tensor.size(0), 1, -1).to(device)

        # Feed the single image sequence to the RNN model
        denoised_image_flat = model(rnn_input)

        # Reshape the output to an image tensor (for CIFAR-10 size)
        denoised_image_tensor = denoised_image_flat.view(1, 1, 32, 32)

        # Display the original random image
        plt.subplot(1, 2, 1)
        plt.imshow(random_image_np, cmap='gray', vmin=0, vmax=1)
        plt.title("Original Random")
        plt.axis('off')

        # Display the denoised image
        denoised_image_np = denoised_image_tensor.squeeze().cpu().numpy()
        plt.subplot(1, 2, 2)
        plt.imshow(denoised_image_np, cmap='gray', vmin=0, vmax=1)
        plt.title("Denoised by RNN")
        plt.axis('off')

    plt.tight_layout()
    plt.show()

# Assume 'model' (the trained RNN model) and 'device' are defined
# Example usage:
# denoise_random_image_rnn(model=model, noising_steps=5, device=device) # noising_steps is now ignored

In [ ]:
# --- 1️⃣ Move model and all parameters safely ---
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
for param in model.parameters():
    param.data = param.data.to(device)

# --- 2️⃣ Denoise function ---
import matplotlib.pyplot as plt
import numpy as np
import torch

def denoise_random_image_rnn(model, device):
    model.eval()
    model.to(device)  # ensure model on device

    plt.figure(figsize=(2, 2))

    with torch.no_grad():
        # Random image ~ Beta(0.5, 0.5)
        random_image_np = np.random.beta(0.5, 0.5, size=(32, 32))
        random_image_tensor = (
            torch.tensor(random_image_np, dtype=torch.float32)
            .unsqueeze(0)
            .unsqueeze(0)
            .to(device)
        )

        # Prepare input
        rnn_input = random_image_tensor.view(random_image_tensor.size(0), 1, -1)

        # 🔥 feed through model
        denoised_image_flat = model(rnn_input)

        # Bring result back to CPU for visualization
        denoised_image_tensor = denoised_image_flat.view(1, 1, 32, 32).cpu()

        # Plot
        plt.subplot(1, 2, 1)
        plt.imshow(random_image_np, cmap="gray", vmin=0, vmax=1)
        plt.title("Original Random")
        plt.axis("off")

        plt.subplot(1, 2, 2)
        plt.imshow(denoised_image_tensor.squeeze().numpy(), cmap="gray", vmin=0, vmax=1)
        plt.title("Denoised by RNN")
        plt.axis("off")

    plt.tight_layout()
    plt.show()

# --- 3️⃣ Run ---
for name, param in model.named_parameters():
    print(name, param.device)

denoise_random_image_rnn(model, device)


In [ ]:
def denoise_random_image_rnn(model, device):
    model.eval()
    plt.figure(figsize=(2, 2))

    with torch.no_grad():
        # Random noisy image
        random_image_np = np.random.beta(0.5, 0.5, size=(32, 32))
        random_image_tensor = torch.tensor(random_image_np, dtype=torch.float32).unsqueeze(0).unsqueeze(0).to(device)

        # Reshape for RNN input
        rnn_input = random_image_tensor.view(random_image_tensor.size(0), 1, -1).to(device)

        # Feed through model
        denoised_image_flat = model(rnn_input)

        # Move output to CPU for plotting
        denoised_image_tensor = denoised_image_flat.view(1, 1, 32, 32).cpu()

        # Plot
        plt.subplot(1, 2, 1)
        plt.imshow(random_image_np, cmap='gray')
        plt.title("Original Random")
        plt.axis('off')

        plt.subplot(1, 2, 2)
        plt.imshow(denoised_image_tensor.squeeze().numpy(), cmap='gray')
        plt.title("Denoised by RNN")
        plt.axis('off')

    plt.tight_layout()
    plt.show()
denoise_random_image_rnn(model, device)